# Step 5 — Candidate Union + Cheap Filtering

**What changed vs. the version you gave me, and why (roughly in order of severity):**

1. **This would OOM on real data, not just run slowly.** `sklearn.metrics.pairwise.linear_kernel` forces
   `dense_output=True` internally — for a 5,000-row query chunk against a multi-million-row S2+S3 database,
   that's a dense `(5000, 5_000_000)` array, i.e. on the order of 200 GB. This isn't a performance nitpick,
   it's the difference between the script running and crashing outright at 50L+ scale. Fixed by keeping the
   query x database product as a sparse matrix throughout, and extracting each row's top-K only from its
   actual nonzero entries (standard technique for large-scale sparse cosine blocking).
2. **A brand-new `TfidfVectorizer` was being fit from scratch on every run — and independently for train vs.
   test.** That's wasted compute (Step 2 already fit and saved this), and more importantly a correctness
   issue: fitting a separate vectorizer per mode means train and test candidates live in *different* TF-IDF
   vector spaces, with no guarantee scores are comparable between them. This notebook reuses the shared,
   already-fitted vectorizer's precomputed sparse matrices from Step 2 directly - nothing gets refit here.
3. **File-glob / filename mismatches**, same class of issue as the last two steps — updated to
   `part_*_meta.parquet` and the `_candidates` suffix Step 3b actually writes.
4. **`bge[bge.entity_id == s1_id]` inside the per-query loop is an O(n) table scan repeated for every S1
   entity** — replaced with a single `groupby` computed once before the loop.
5. **`np.argpartition(sim[i], -TOP_K)` would crash outright if the database has fewer than `TOP_K` candidates**
   (easy to hit on a small dev subset) — guarded.
6. **Duplicate-row handling changed.** The original *dropped* the other blocker's score whenever a pair was
   found by more than one blocker (kept only the row with the best `bge_cosine`, discarding that pair's
   `tfidf_score` even when it existed), and only did this for train. Replaced with a proper merge — one row
   per `(entity_id, candidate_id)` pair with every blocker's score kept — applied to **both** train and test,
   since leaving test un-merged would mean the same physical pair gets scored twice downstream with two
   different incomplete feature patterns the model never saw during training. I'm overriding the explicit
   train-only asymmetry you had here — if there was a specific downstream reason for it, worth telling me,
   since this change assumes there wasn't one.
7. **Unions in char-level TF-IDF candidates too** (typo/transliteration coverage) since Step 2 already
   computed this for free — toggle `USE_CHAR_TFIDF = False` to go back to word-only.
8. Added a lightweight print of how much the country filter drops. Worth checking country-label consistency
   across sources before trusting an exact-match filter fully - a true match dropped here can't be
   recovered later.

One thing this script still *doesn't* do: your earlier short-pipeline sketch had three blockers (BGE + TF-IDF
+ "simple blocking"). This step only unions the first two - the rule-based blocker isn't implemented here,
presumably it's meant to be a separate step.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp

In [ ]:
# ======================================================
# CONFIG
# ======================================================

MODE = "train"  # "train" or "test" - run this notebook once for each

TOP_K = 20
QUERY_CHUNK = 5000

S1_DIR = Path(f"processed/source1_{MODE}_embeddings")
S2_DIR = Path(f"processed/source2_{MODE}_embeddings")
S3_DIR = Path(f"processed/source3_{MODE}_embeddings")

BGE_DIR = Path(f"candidates/{MODE}")
OUT_DIR = Path(f"filtered_candidates/{MODE}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Union in char-level TF-IDF candidates alongside word-level - Step 2
# already computed both, so this costs nothing extra to include.
USE_CHAR_TFIDF = True

In [ ]:
# ======================================================
# LOAD PRECOMPUTED TF-IDF (reused from Step 2 - nothing is refit here)
# ======================================================

def load_folder_tfidf(folder, use_char):
    """Concatenate every chunk in `folder` into one aligned (meta, word_matrix, char_matrix)."""
    meta_files = sorted(folder.glob("part_*_meta.parquet"))
    if not meta_files:
        raise FileNotFoundError(f"No 'part_*_meta.parquet' files found in {folder}.")

    meta_frames, word_mats, char_mats = [], [], []

    for meta_file in meta_files:
        stem = meta_file.name.replace("_meta.parquet", "")
        word_file = folder / f"{stem}_tfidf_word.npz"
        char_file = folder / f"{stem}_tfidf_char.npz"

        meta = pd.read_parquet(meta_file, columns=["entity_id", "country"])
        meta_frames.append(meta)
        word_mats.append(sp.load_npz(word_file))

        if use_char:
            if not char_file.exists():
                raise FileNotFoundError(
                    f"{char_file} not found - re-run Step 2 with USE_CHAR_TFIDF=True, "
                    f"or set USE_CHAR_TFIDF=False here."
                )
            char_mats.append(sp.load_npz(char_file))

    meta = pd.concat(meta_frames, ignore_index=True)
    word_matrix = sp.vstack(word_mats).tocsr()
    char_matrix = sp.vstack(char_mats).tocsr() if use_char else None

    assert word_matrix.shape[0] == len(meta), f"TF-IDF/metadata row mismatch in {folder}"
    if use_char:
        assert char_matrix.shape[0] == len(meta), f"Char TF-IDF/metadata row mismatch in {folder}"

    return meta, word_matrix, char_matrix


def load_single_file_tfidf(meta_file, use_char):
    stem = meta_file.name.replace("_meta.parquet", "")
    folder = meta_file.parent
    meta = pd.read_parquet(meta_file, columns=["entity_id", "country"])
    word_matrix = sp.load_npz(folder / f"{stem}_tfidf_word.npz").tocsr()
    char_matrix = sp.load_npz(folder / f"{stem}_tfidf_char.npz").tocsr() if use_char else None

    assert word_matrix.shape[0] == len(meta), f"TF-IDF/metadata row mismatch in {meta_file}"
    if use_char:
        assert char_matrix.shape[0] == len(meta), f"Char TF-IDF/metadata row mismatch in {meta_file}"

    return meta, word_matrix, char_matrix

In [ ]:
# ======================================================
# LOAD DATABASE (S2 + S3), ONCE
# ======================================================

print("Loading S2 + S3 database (TF-IDF reused from Step 2, not refit)...")

db_meta_parts, db_word_parts, db_char_parts = [], [], []
for folder in [S2_DIR, S3_DIR]:
    meta, word_mat, char_mat = load_folder_tfidf(folder, USE_CHAR_TFIDF)
    db_meta_parts.append(meta)
    db_word_parts.append(word_mat)
    if USE_CHAR_TFIDF:
        db_char_parts.append(char_mat)

db_meta = pd.concat(db_meta_parts, ignore_index=True)
db_word_matrix = sp.vstack(db_word_parts).tocsr()
db_char_matrix = sp.vstack(db_char_parts).tocsr() if USE_CHAR_TFIDF else None
db_ids = db_meta["entity_id"].to_numpy()

print(f"Database: {len(db_meta):,} S2+S3 records")

In [ ]:
# ======================================================
# SPARSE TOP-K (never materializes a dense query x database matrix)
# ======================================================

def sparse_topk_per_query(sim_block, top_k):
    """
    sim_block: scipy.sparse CSR matrix, shape (n_queries, n_db) - the RAW
    sparse dot product (query_matrix @ db_matrix.T), not run through
    sklearn's linear_kernel (which forces dense_output=True and would
    OOM at this scale). Returns, per row, the (indices, scores) of its
    top_k highest entries, computed only over that row's nonzero values.
    """
    sim_block = sim_block.tocsr()
    results = []
    for r in range(sim_block.shape[0]):
        start, end = sim_block.indptr[r], sim_block.indptr[r + 1]
        row_idx = sim_block.indices[start:end]
        row_val = sim_block.data[start:end]

        if len(row_val) == 0:
            results.append((np.array([], dtype=np.int64), np.array([], dtype=np.float32)))
            continue

        k = min(top_k, len(row_val))  # guards against a database smaller than top_k
        top_local = np.argpartition(row_val, -k)[-k:]
        top_local = top_local[np.argsort(row_val[top_local])[::-1]]
        results.append((row_idx[top_local], row_val[top_local]))
    return results

In [ ]:
# ======================================================
# PROCESS EACH S1 CHUNK
# ======================================================

s1_files = sorted(S1_DIR.glob("part_*_meta.parquet"))
print(f"Found {len(s1_files)} Source 1 ({MODE}) chunks")

for s1_file in s1_files:

    out_file = OUT_DIR / s1_file.name.replace("_meta", "_filtered")
    if out_file.exists():
        print(f"[SKIP] {s1_file.name}")
        continue

    print(f"[PROCESS] {s1_file.name}")
    s1_meta, s1_word_matrix, s1_char_matrix = load_single_file_tfidf(s1_file, USE_CHAR_TFIDF)
    s1_ids = s1_meta["entity_id"].to_numpy()

    bge_file = BGE_DIR / s1_file.name.replace("_meta", "_candidates")
    if bge_file.exists():
        bge = pd.read_parquet(bge_file)
    else:
        bge = pd.DataFrame(columns=["entity_id", "candidate_id", "rank", "bge_cosine"])
    bge_by_entity = bge.groupby("entity_id")

    rows = []

    for start in range(0, s1_word_matrix.shape[0], QUERY_CHUNK):
        end = min(start + QUERY_CHUNK, s1_word_matrix.shape[0])

        # Sparse-sparse matmul stays sparse - this is the fix for the OOM issue above.
        word_sim = s1_word_matrix[start:end] @ db_word_matrix.T
        word_topk = sparse_topk_per_query(word_sim, TOP_K)

        char_topk = None
        if USE_CHAR_TFIDF:
            char_sim = s1_char_matrix[start:end] @ db_char_matrix.T
            char_topk = sparse_topk_per_query(char_sim, TOP_K)

        for i in range(end - start):
            s1_id = s1_ids[start + i]

            # ---- BGE ----
            if s1_id in bge_by_entity.groups:
                for _, r in bge_by_entity.get_group(s1_id).iterrows():
                    rows.append({
                        "entity_id": s1_id, "candidate_id": r["candidate_id"],
                        "bge_cosine": r["bge_cosine"], "tfidf_word_score": np.nan,
                        "tfidf_char_score": np.nan, "source": "BGE",
                    })

            # ---- TF-IDF word ----
            idx_arr, val_arr = word_topk[i]
            for cand_pos, score in zip(idx_arr, val_arr):
                rows.append({
                    "entity_id": s1_id, "candidate_id": db_ids[cand_pos],
                    "bge_cosine": np.nan, "tfidf_word_score": float(score),
                    "tfidf_char_score": np.nan, "source": "TFIDF_WORD",
                })

            # ---- TF-IDF char ----
            if char_topk is not None:
                idx_arr, val_arr = char_topk[i]
                for cand_pos, score in zip(idx_arr, val_arr):
                    rows.append({
                        "entity_id": s1_id, "candidate_id": db_ids[cand_pos],
                        "bge_cosine": np.nan, "tfidf_word_score": np.nan,
                        "tfidf_char_score": float(score), "source": "TFIDF_CHAR",
                    })

    df = pd.DataFrame(rows)

    # ==========================================
    # MERGE DUPLICATE (entity_id, candidate_id) ROWS ACROSS BLOCKERS
    # ==========================================
    # One row per pair, keeping every blocker's score that found it -
    # see point 6 in the markdown above for why this replaces the
    # train-only dedup and applies to both modes now.
    if len(df):
        df = df.groupby(["entity_id", "candidate_id"], as_index=False).agg({
            "bge_cosine": "max",
            "tfidf_word_score": "max",
            "tfidf_char_score": "max",
            "source": lambda s: "+".join(sorted(set(s))),
        })

    # ==========================================
    # CHEAP FILTER: same country only
    # ==========================================
    s1_country = s1_meta[["entity_id", "country"]]
    db_country = db_meta[["entity_id", "country"]].rename(
        columns={"entity_id": "candidate_id", "country": "candidate_country"}
    )

    before = len(df)
    df = df.merge(s1_country, on="entity_id", how="left")
    df = df.merge(db_country, on="candidate_id", how="left")
    df = df[df["country"] == df["candidate_country"]]
    dropped = before - len(df)
    if before:
        print(f"  country filter: dropped {dropped:,}/{before:,} candidates ({100 * dropped / before:.1f}%)")

    df = df.drop(columns=["country", "candidate_country"])
    df.to_parquet(out_file, index=False)
    print(f"[DONE] {s1_file.name}: {df['entity_id'].nunique():,} S1 entities, {len(df):,} candidate rows")

(OUT_DIR / "_SUCCESS").touch()
print(f"Step 5 completed for {MODE}.")